# Medical Visit Summary - Model Comparison

**Purpose:** Compare AI models (OpenAI vs Gemini) on medical visit summaries.

**Workflow:**
1. Run cells 1-3 (setup)
2. Run cell 4 (comparison)
3. Review output and decide

---


In [31]:
import os
import sys
import json
import time
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai as genai

# Load .env file
backend_dir = Path(os.getcwd()).parent
load_dotenv(backend_dir / '.env')

# Configure APIs
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
sys.path.insert(0, str(backend_dir))

print("✓ Environment loaded")


✓ Environment loaded


In [42]:
# Import both prompts for comparison
from utils.prompts.medical_summary import MEDICAL_SUMMARY_PROMPT as PROMPT_V1
from utils.prompts.medical_summaryv2 import MEDICAL_SUMMARY_PROMPT as PROMPT_V2

# Use V2 by default
MEDICAL_SUMMARY_PROMPT = PROMPT_V2

print("✓ Prompts loaded (using V2 by default)")
print("  V1: Original prompt (comprehensive)")
print("  V2: New prompt (accuracy-focused)")


✓ Prompts loaded (using V2 by default)
  V1: Original prompt (comprehensive)
  V2: New prompt (accuracy-focused)


## Option A: Load Text Transcript


In [43]:
# Load text transcript
transcript_file = 'sample_visit_1.txt'  # Change to sample_visit_2.txt if needed
with open(f'transcripts/{transcript_file}', 'r') as f:
    transcript = f.read()

print(f"✓ Loaded transcript ({len(transcript)} chars)")
print(f"\nPreview:\n{transcript[:200]}...")


✓ Loaded transcript (1944 chars)

Preview:
Doctor: Good morning! How are you feeling today?

Patient: Hi Doctor. Not so great, to be honest. I've been having this really bad chest pain for about three days now.

Doctor: I see. Can you describe...


## Option B: Load Audio File (Alternative)


In [44]:
# Transcribe audio file using Whisper
def transcribe_audio(audio_file):
    """Transcribe audio file using OpenAI Whisper API."""
    print(f"Transcribing {audio_file}...")
    
    client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    with open(f'audio/{audio_file}', 'rb') as f:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            language="en"
        )
    
    transcript = response.text
    print(f"✓ Transcribed ({len(transcript)} chars)")
    return transcript

# READY TO USE: Your audio is downloaded! Uncomment these 3 lines:
audio_file = '7f935bd1-6976-4fc7-b9a3-08aa59211c34.m4a'  # Longest duration (358 KB)
transcript = transcribe_audio(audio_file)
print(f"\nTranscript preview:\n{transcript[:300]}...")


Transcribing 7f935bd1-6976-4fc7-b9a3-08aa59211c34.m4a...
✓ Transcribed (332 chars)

Transcript preview:
Patient, doctor, I've been having a headache for the last three days. It's mostly on the left side, and it gets worse in the evening. Doctor, okay, any nausea, vomiting, or sensitivity to light? Patient, no vomiting, but I do feel a little dizzy sometimes. Doctor, all right, do you have any fever or...


## Compare Models


In [39]:
def compare_models(transcript, models, temperature=0.2):
    """Compare multiple models on the same transcript."""
    
    print("="*100)
    print("MODEL COMPARISON")
    print("="*100)
    print(f"Transcript: {len(transcript)} chars | Temperature: {temperature}\n")
    
    results = []
    
    # Test each model
    for model in models:
        print(f"Testing {model}...", end=" ")
        start_time = time.time()
        
        try:
            # Format prompt
            prompt = MEDICAL_SUMMARY_PROMPT.format(
                current_datetime=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                transcript=transcript
            )
            
            # Call API based on model type
            if model.startswith("gemini"):
                gemini_model = genai.GenerativeModel(model)
                response = gemini_model.generate_content(
                    prompt,
                    generation_config=genai.types.GenerationConfig(temperature=temperature)
                )
                text = response.text.strip()
                start_idx = text.find("{")
                end_idx = text.rfind("}")
                result = json.loads(text[start_idx:end_idx + 1])
            else:
                client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
                response = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": prompt}],
                    temperature=temperature,
                    response_format={"type": "json_object"}
                )
                result = json.loads(response.choices[0].message.content)
            
            latency = time.time() - start_time
            results.append({'model': model, 'data': result, 'latency': latency})
            print(f"✅ ({latency:.1f}s)")
            
        except Exception as e:
            print(f"❌ {str(e)[:50]}")
    
    if not results:
        return
    
    # Print metrics table
    print("\n" + "="*100)
    print("METRICS")
    print("="*100)
    print(f"{'Model':<30} {'Speed':<10} {'Words':<8} {'Dx':<5} {'Meds':<6} {'Actions':<8}")
    print("-"*100)
    
    for r in results:
        words = len(r['data'].get('summary', '').split())
        dx = len(r['data'].get('key_diagnoses', []))
        meds = len(r['data'].get('medications', []))
        actions = len(r['data'].get('action_items', []))
        print(f"{r['model']:<30} {r['latency']:.1f}s{'':<6} {words:<8} {dx:<5} {meds:<6} {actions:<8}")
    
    # Print summaries
    print("\n" + "="*100)
    print("SUMMARIES")
    print("="*100)
    for r in results:
        print(f"\n[{r['model']}] ({len(r['data']['summary'].split())} words)")
        print("-"*100)
        print(r['data']['summary'])
    
    # Print action items
    print("\n" + "="*100)
    print("ACTION ITEMS")
    print("="*100)
    for r in results:
        print(f"\n[{r['model']}]")
        for i, action in enumerate(r['data'].get('action_items', []), 1):
            print(f"  {i}. {action}")
    
    # Summary
    print("\n" + "="*100)
    print("QUICK TAKEAWAYS")
    print("="*100)
    fastest = min(results, key=lambda x: x['latency'])
    print(f"⚡ Fastest: {fastest['model']} ({fastest['latency']:.1f}s)")
    most_concise = min(results, key=lambda x: len(x['data']['summary'].split()))
    print(f"📏 Most concise: {most_concise['model']} ({len(most_concise['data']['summary'].split())} words)")
    print("\n💡 Now manually review: Which is more accurate? Which is clearer?")
    print("="*100)

print("✓ Function ready")


✓ Function ready


In [45]:
# Run comparison on transcript (from text or audio)
models_to_test = [
    "gemini-2.0-flash-lite",  # Your production model
    "gpt-4o-mini",            # OpenAI alternative
]

compare_models(transcript, models_to_test, temperature=0.2)


MODEL COMPARISON
Transcript: 332 chars | Temperature: 0.2

Testing gemini-2.0-flash-lite... ❌ 429 You exceeded your current quota, please check 
Testing gpt-4o-mini... ✅ (5.0s)

METRICS
Model                          Speed      Words    Dx    Meds   Actions 
----------------------------------------------------------------------------------------------------
gpt-4o-mini                    5.0s       32       1     1      0       

SUMMARIES

[gpt-4o-mini] (32 words)
----------------------------------------------------------------------------------------------------
The patient reported having a headache for the last three days, primarily on the left side, worsening in the evening. The patient experienced dizziness but no vomiting, nausea, fever, or recent cold.

ACTION ITEMS

[gpt-4o-mini]

QUICK TAKEAWAYS
⚡ Fastest: gpt-4o-mini (5.0s)
📏 Most concise: gpt-4o-mini (32 words)

💡 Now manually review: Which is more accurate? Which is clearer?


In [ ]:
## Single Model Test: GPT-4o-mini

Test just one model and see the raw transcript + output


In [46]:
# Test GPT-4o-mini on the transcript
print("="*100)
print("RAW TRANSCRIPT")
print("="*100)
print(transcript)
print("\n" + "="*100)
print("GPT-4o-mini OUTPUT")
print("="*100)

# Generate summary with GPT-4o-mini
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
prompt = MEDICAL_SUMMARY_PROMPT.format(
    current_datetime=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    transcript=transcript
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "system", "content": prompt}],
    temperature=0.2,
    response_format={"type": "json_object"}
)

result = json.loads(response.choices[0].message.content)

# Print the output nicely
print(f"\n📋 Title: {result.get('title')}")
print(f"\n📝 Summary ({len(result.get('summary', '').split())} words):")
print(result.get('summary'))

print(f"\n🔍 Key Diagnoses:")
for d in result.get('key_diagnoses', []):
    print(f"  • {d}")

print(f"\n💊 Medications:")
for m in result.get('medications', []):
    print(f"  • {m}")

print(f"\n✅ Action Items:")
for a in result.get('action_items', []):
    print(f"  • {a}")

print(f"\n❓ Questions for Next Visit:")
for q in result.get('questions_next_visit', []):
    print(f"  • {q}")

print("\n" + "="*100)


RAW TRANSCRIPT
Patient, doctor, I've been having a headache for the last three days. It's mostly on the left side, and it gets worse in the evening. Doctor, okay, any nausea, vomiting, or sensitivity to light? Patient, no vomiting, but I do feel a little dizzy sometimes. Doctor, all right, do you have any fever or recent cold? Patient, no fever.

GPT-4o-mini OUTPUT

📋 Title: Headache Evaluation

📝 Summary (32 words):
The patient reported having a headache for the last three days, primarily on the left side, worsening in the evening. The patient experienced dizziness but no nausea, vomiting, fever, or recent cold.

🔍 Key Diagnoses:
  • Not mentioned in transcript

💊 Medications:
  • Not mentioned in transcript

✅ Action Items:

❓ Questions for Next Visit:
  • What other symptoms have developed since this visit?
  • Have you noticed any triggers for your headaches?
  • How has the severity of your headaches changed?



## Prompt Comparison: V1 vs V2

Compare the two different prompts using GPT-4o-mini

**V1:** Original prompt (comprehensive, 90-word minimum)  
**V2:** New prompt (accuracy-focused, safety rules)


In [47]:
# Compare both prompts using GPT-4o-mini
print("="*100)
print("PROMPT COMPARISON: V1 vs V2 (using GPT-4o-mini)")
print("="*100)
print(f"\nTranscript: {len(transcript)} chars\n")

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
results = {}

# Test both prompts
for name, prompt_template in [("V1 (Original)", PROMPT_V1), ("V2 (New)", PROMPT_V2)]:
    print(f"Testing {name}...", end=" ")
    
    prompt = prompt_template.format(
        current_datetime=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        transcript=transcript
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": prompt}],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    results[name] = result
    print("✅")

# Print comparison
print("\n" + "="*100)
print("METRICS")
print("="*100)
print(f"{'Prompt':<20} {'Words':<10} {'Dx':<5} {'Meds':<6} {'Actions':<8}")
print("-"*100)

for name, result in results.items():
    words = len(result.get('summary', '').split())
    dx = len(result.get('key_diagnoses', []))
    meds = len(result.get('medications', []))
    actions = len(result.get('action_items', []))
    print(f"{name:<20} {words:<10} {dx:<5} {meds:<6} {actions:<8}")

# Compare summaries
print("\n" + "="*100)
print("SUMMARIES")
print("="*100)

for name, result in results.items():
    words = len(result.get('summary', '').split())
    print(f"\n[{name}] ({words} words)")
    print("-"*100)
    print(result.get('summary', 'N/A'))

# Compare key diagnoses
print("\n" + "="*100)
print("KEY DIAGNOSES")
print("="*100)

for name, result in results.items():
    print(f"\n[{name}]")
    for d in result.get('key_diagnoses', []):
        print(f"  • {d}")

# Compare medications
print("\n" + "="*100)
print("MEDICATIONS")
print("="*100)

for name, result in results.items():
    print(f"\n[{name}]")
    for m in result.get('medications', []):
        print(f"  • {m}")

# Compare action items
print("\n" + "="*100)
print("ACTION ITEMS")
print("="*100)

for name, result in results.items():
    print(f"\n[{name}]")
    actions = result.get('action_items', [])
    if actions:
        for a in actions:
            print(f"  • {a}")
    else:
        print("  (none)")

# Analysis
print("\n" + "="*100)
print("QUALITY ANALYSIS")
print("="*100)

v1_words = len(results["V1 (Original)"].get('summary', '').split())
v2_words = len(results["V2 (New)"].get('summary', '').split())

print(f"\n📏 Length:")
print(f"  V1: {v1_words} words")
print(f"  V2: {v2_words} words")

print(f"\n🎯 Key Differences:")
print(f"  V1: Aims for comprehensive summary (90-word minimum)")
print(f"  V2: Aims for accuracy (only stated facts, no minimum)")

print(f"\n💡 Evaluation Points:")
print(f"  1. Which is more accurate to the transcript?")
print(f"  2. Which captures all critical information?")
print(f"  3. Which is clearer for patients to understand?")
print(f"  4. Which follows 'state only what's in transcript' better?")

print("\n" + "="*100)


PROMPT COMPARISON: V1 vs V2 (using GPT-4o-mini)

Transcript: 332 chars

Testing V1 (Original)... ✅
Testing V2 (New)... ✅

METRICS
Prompt               Words      Dx    Meds   Actions 
----------------------------------------------------------------------------------------------------
V1 (Original)        47         1     0      2       
V2 (New)             33         1     1      0       

SUMMARIES

[V1 (Original)] (47 words)
----------------------------------------------------------------------------------------------------
The patient presented with a left-sided headache lasting 3 days, worsening in the evening, and occasional dizziness. The doctor assessed the symptoms and did not find any signs of fever or cold. No medications were prescribed during this visit. Follow-up is recommended if symptoms persist or worsen.

[V2 (New)] (33 words)
----------------------------------------------------------------------------------------------------
The patient reported having a headache for

## Simple Prompt Comparison (No Truncation)

Shorter output showing just the summaries side-by-side


In [48]:
# Quick comparison - won't get truncated
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("🔬 QUICK PROMPT COMPARISON (GPT-4o-mini)\n")

results = {}
for name, prompt_template in [("V1", PROMPT_V1), ("V2", PROMPT_V2)]:
    prompt = prompt_template.format(
        current_datetime=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        transcript=transcript
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": prompt}],
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    results[name] = json.loads(response.choices[0].message.content)

# Show summaries
print("📝 SUMMARIES:")
print("-" * 80)
for name, result in results.items():
    words = len(result.get('summary', '').split())
    print(f"\n[{name}] ({words} words)")
    print(result.get('summary'))

# Show action items
print("\n\n✅ ACTION ITEMS:")
print("-" * 80)
for name, result in results.items():
    print(f"\n[{name}]")
    actions = result.get('action_items', [])
    if actions:
        for a in actions:
            print(f"  • {a}")
    else:
        print("  (none)")

# Quick verdict
v1_words = len(results["V1"].get('summary', '').split())
v2_words = len(results["V2"].get('summary', '').split())

print(f"\n\n🎯 QUICK COMPARISON:")
print(f"  V1: {v1_words} words (comprehensive)")
print(f"  V2: {v2_words} words (accuracy-focused)")
print(f"\n  💡 V2 is shorter because it only states explicit facts.")
print(f"     V1 is longer for completeness (90-word minimum).")


🔬 QUICK PROMPT COMPARISON (GPT-4o-mini)

📝 SUMMARIES:
--------------------------------------------------------------------------------

[V1] (44 words)
The patient presented with a headache on the left side lasting for three days, worsening in the evening, and occasional dizziness. The doctor assessed the situation and noted no fever or other alarming symptoms. A follow-up appointment is recommended if symptoms persist or worsen.

[V2] (39 words)
The patient reported having a headache for the last three days, primarily on the left side, worsening in the evening. The patient experienced some dizziness but no vomiting or sensitivity to light. There was no fever or recent cold.


✅ ACTION ITEMS:
--------------------------------------------------------------------------------

[V1]
  • Schedule a follow-up appointment if symptoms persist or worsen

[V2]
  (none)


🎯 QUICK COMPARISON:
  V1: 44 words (comprehensive)
  V2: 39 words (accuracy-focused)

  💡 V2 is shorter because it only states 